"""
Pretrained Transfer Learning Experiment - Di et al. (2025).

Task description (English):
==========================
This experiment replicates and extends Di et al. (2025) Section 4.1 with a
two-stage "pretrained" transfer-learning pipeline:

Stage 1 - Source pretraining with early stopping:
  - Train a 6-layer CNN on the source dataset.
  - Early stopping based on source validation loss.
  - After early stopping, continue training for POST_EARLY_STOP_EPOCHS (=10)
    additional epochs for control.
  - Log per epoch: train loss, source test accuracy, source validation loss,
    target validation loss, and projection correlation (PC) for all layers.
  - PC and target validation loss are computed on SAMPLE_SIZES elements drawn
    NUM_REPLICATIONS times from the target dataset (confidence intervals).
  - Draw all plots; save the pretrained model.

Stage 2 - Target fine-tuning with early stopping:
  - Fine-tune the pretrained model on the target dataset using SAMPLE_SIZES.
  - Early stopping based on target validation loss.


In [3]:
"""
- Log per epoch: train loss, test accuracy, target validation loss, and PC
    for all layers.
  - Draw all plots; save the fine-tuned model.

Results are written to a directory named after this script
(e.g. pretrained/) with a timestamp appended to each result filename.
"""

import argparse
import sys
import os
import json
import time
import glob
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, random_split

try:
    from .models import CIFAR6CNN, create_transferred_model, set_trainable_layers
    from .projection_correlation import projection_correlation
    from .layer_extractor import extract_layer_activations
    from .dataset_splits import (
        split_cifar_imbalance,
        split_cifar_balance,
        generate_synthetic_transfer_dataset,
        make_dataloaders,
        CIFAR_SOURCE_CLASSES,
        CIFAR_TARGET_CLASSES,
    )
    from .di_algorithm import (
        compute_pc_profile,
        estimate_phase_transition,
        forward_adding_layer_selection,
        train_target_candidate,
        default_eval_loss,
    )
except ImportError:
    from models import CIFAR6CNN, create_transferred_model, set_trainable_layers
    from projection_correlation import projection_correlation
    from layer_extractor import extract_layer_activations
    from dataset_splits import (
        split_cifar_imbalance,
        split_cifar_balance,
        generate_synthetic_transfer_dataset,
        make_dataloaders,
        CIFAR_SOURCE_CLASSES,
        CIFAR_TARGET_CLASSES,
    )
    from di_algorithm import (
        compute_pc_profile,
        estimate_phase_transition,
        forward_adding_layer_selection,
        train_target_candidate,
        default_eval_loss,
    )

# Named constant: extra epochs after early stopping in Stage 1
POST_EARLY_STOP_EPOCHS = 10

# Default patience for early stopping
EARLY_STOP_PATIENCE = 10


ModuleNotFoundError: No module named 'models'

In [ ]:
def _timestamp() -> str:
    """Return a compact timestamp string YYYYMMDD_HHMMSS."""
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def _results_dir(script_name: str) -> str:
    """Return results directory named after the script file (without .py)."""
    base = os.path.splitext(os.path.basename(script_name))[0]
    d = os.path.join(os.path.dirname(os.path.abspath(script_name)), base)
    os.makedirs(d, exist_ok=True)
    return d


def _save_json(path: str, data: dict) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)
    print(f"Results saved to: {path}")


def _select_target_subset(
    target_dataset: Tuple[torch.Tensor, torch.Tensor],
    n_samples: int,
    seed: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Select a fixed subset of n_samples from the target dataset."""
    x, y = target_dataset
    n_total = x.shape[0]
    if n_samples >= n_total:
        return x, y
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(n_total, generator=generator)[:n_samples]
    return x[indices], y[indices]


def _make_subset_loader(
    dataset: Tuple[torch.Tensor, torch.Tensor],
    batch_size: int,
    shuffle: bool = False,
) -> DataLoader:
    """Build a DataLoader from a (x, y) tensor tuple."""
    x, y = dataset
    tensor_dataset = torch.utils.data.TensorDataset(x, y)
    return DataLoader(tensor_dataset, batch_size=batch_size, shuffle=shuffle)



def _compute_pc_with_ci(


In [ ]:
model: nn.Module,
    target_dataset: Tuple[torch.Tensor, torch.Tensor],
    sample_sizes: List[int],
    num_replications: int,
    device: torch.device,
    base_seed: int = 42,
) -> Dict[int, Dict[str, List[float]]]:
    """
    Compute projection correlation for all layers across sample sizes and
    replications. Returns dict: {n: {layer: [rep1, rep2, ...]}}.
    """
    results: Dict[int, Dict[str, List[float]]] = {}
    layer_names = CIFAR6CNN.LAYER_NAMES

    for n_s in sample_sizes:
        results[n_s] = {layer: [] for layer in layer_names}
        for rep in range(num_replications):
            x_sub, y_sub = _select_target_subset(
                target_dataset, n_s, seed=base_seed + rep
            )
            sub_loader = _make_subset_loader((x_sub, y_sub), batch_size=64, shuffle=False)
            prof = compute_pc_profile(
                model, sub_loader,
                device=device, max_samples=n_s,
                verbose=False,
            )
            for layer, val in prof['omega'].items():
                results[n_s][layer].append(float(val))
    return results


def _summarize_pc(
    pc_results: Dict[int, Dict[str, List[float]]],
    layer_names: List[str],
) -> str:
    """Build a summary string of PC means +/- stds for the largest sample size."""
    if not pc_results:
        return ""
    last_n = max(pc_results.keys())
    parts = []
    for layer in layer_names:
        vals = pc_results[last_n][layer]
        if vals:
            parts.append(f"{layer}={np.mean(vals):.3f}±{np.std(vals):.3f}")
    return " | ".join(parts)


def _plot_pc_profiles(


In [ ]:
pc_results: Dict[int, Dict[str, List[float]]],
    layer_names: List[str],
    save_path: str,
    title_suffix: str = "",
) -> None:
    """Plot PC profiles with error bars for all sample sizes."""
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 5))
    markers = ['o-', 's--', '^:']
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    layer_idx = np.arange(1, len(layer_names) + 1)

    for idx, n_s in enumerate(sorted(pc_results.keys())):
        means = [np.mean(pc_results[n_s][l]) for l in layer_names]
        stds = [np.std(pc_results[n_s][l]) for l in layer_names]
        plt.errorbar(
            layer_idx, means, yerr=stds,
            fmt=markers[idx % len(markers)],
            color=colors[idx % len(colors)],
            capsize=4, linewidth=2, markersize=7,
            label=f"n={n_s}",
        )

    plt.xticks(layer_idx, layer_names, rotation=45)
    plt.xlabel("Layer", fontsize=11, fontweight='bold')
    plt.ylabel("Projection Correlation (PC)", fontsize=11, fontweight='bold')
    plt.title(f"PC Generality Profiles{title_suffix}", fontsize=12, fontweight='bold')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Plot saved to: {save_path}")


def _plot_training_curves(
    history: List[dict],
    save_path: str,
    title_suffix: str = "",
) -> None:
    """Plot train loss, val loss, and accuracy over epochs."""
    import matplotlib.pyplot as plt

    epochs = [h['epoch'] for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_loss = [h.get('val_loss', np.nan) for h in history]
    test_acc = [h.get('test_acc', np.nan) for h in history]
    target_val_loss = [h.get('target_val_loss', np.nan) for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, train_loss, 'o-', color='#1f77b4', linewidth=2, markersize=4)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Train Loss")
    axes[0].set_title("Training Loss")
    axes[0].grid(True, linestyle=':', alpha=0.6)

    axes[1].plot(epochs, val_loss, 's-', color='#d62728', linewidth=2, markersize=4)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Val Loss")
    axes[1].set_title("Validation Loss")
    axes[1].grid(True, linestyle=':', alpha=0.6)

    axes[2].plot(epochs, test_acc, '^-', color='#2ca02c', linewidth=2, markersize=4)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Test Acc (%)")
    axes[2].set_title("Test Accuracy")
    axes[2].grid(True, linestyle=':', alpha=0.6)

    plt.suptitle(f"Training Curves{title_suffix}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Plot saved to: {save_path}")



def train_with_early_stopping(


In [ ]:
model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_epochs: int,
    patience: int = EARLY_STOP_PATIENCE,
    target_val_loader: Optional[DataLoader] = None,
    target_dataset: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    sample_sizes: Optional[List[int]] = None,
    num_replications: Optional[int] = None,
    prefix: str = "",
    verbose: bool = True,
) -> Tuple[nn.Module, List[dict], int]:
    """
    Train a model with early stopping on validation loss.

    Returns (best_model, history, epochs_trained).
    """
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    history: List[dict] = []

    model.train()
    for ep in range(1, max_epochs + 1):
        # Training step
        tr_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * x.size(0)
        avg_train_loss = tr_loss / len(train_loader.dataset)

        # Source validation
        v_loss, v_acc = default_eval_loss(model, val_loader, criterion, device)

        # Target validation loss (+ PC with CI) if target data provided
        target_val_loss_val = None
        pc_summary = ""
        if target_val_loader is not None and target_dataset is not None:
            target_val_loss_val, _ = default_eval_loss(
                model, target_val_loader, criterion, device
            )
            if sample_sizes is not None and num_replications is not None:
                pc_results = _compute_pc_with_ci(
                    model, target_dataset,
                    sample_sizes=sample_sizes,
                    num_replications=num_replications,
                    device=device,
                    base_seed=42 + ep,
                )
                pc_summary = _summarize_pc(pc_results, CIFAR6CNN.LAYER_NAMES)

        # Logging
        log_parts = [
            f"Epoch {ep:02d}/{max_epochs:02d}",
            f"Train Loss: {avg_train_loss:.4f}",
            f"Source Test Acc: {v_acc * 100:.1f}%",
            f"Source Val Loss: {v_loss:.4f}",
        ]
        if target_val_loss_val is not None:
            log_parts.append(f"Target Val Loss: {target_val_loss_val:.4f}")
        if pc_summary:
            log_parts.append(f"PC: {pc_summary}")
        if verbose:
            print("  " + " | ".join(log_parts))

        history.append({
            'epoch': ep,
            'train_loss': avg_train_loss,
            'val_loss': v_loss,
            'test_acc': v_acc * 100.0,
            'target_val_loss': target_val_loss_val,
            'pc_summary': pc_summary,
        })

        # Early stopping check
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"  Early stopping triggered at epoch {ep} (patience={patience})")
                break

    # Restore best state
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, len(history)



def run_pretrained_experiment(


In [ ]:
case: str = 'imbalance',
    sample_sizes: List[int] = None,
    num_replications: int = 2,
    pretrain_epochs: int = 150,
    target_epochs: int = 100,
    use_synthetic: bool = True,
    device: Optional[torch.device] = None,
    output_dir: Optional[str] = None,
    max_images: Optional[int] = None,
    data_dir: str = './data',
    use_hf: bool = True,
    early_stop_patience: int = EARLY_STOP_PATIENCE,
    post_early_stop_epochs: int = POST_EARLY_STOP_EPOCHS,
):
    if sample_sizes is None:
        sample_sizes = [500, 1000]

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    results_dir = _results_dir(__file__)
    ts = _timestamp()

    print("=" * 70)
    print(f"Pretrained Transfer Learning Experiment: {case.upper()} CASE")
    print(f"Device: {device} | Sample sizes: {sample_sizes} | Replications: {num_replications}")
    print(f"Post early-stop epochs: {post_early_stop_epochs} | Patience: {early_stop_patience}")
    print("=" * 70)

    if use_synthetic:
        splits = generate_synthetic_transfer_dataset(
            n_source=1200, n_target=800,
            num_classes=5 if case == 'imbalance' else 10,
            transition_layer=3 if case == 'imbalance' else 4,
            seed=42,
        )
    else:
        from cifar10_experiment import load_cifar10_raw
        raw_x, raw_y = load_cifar10_raw(
            data_dir=data_dir,
            max_images=max_images,
            use_hf=use_hf,
        )
        if case == 'imbalance':
            splits = split_cifar_imbalance(raw_x, raw_y)
        else:
            splits = split_cifar_balance(raw_x, raw_y)

    loaders = make_dataloaders(splits, batch_size=64)
    target_dataset_full = (splits['target_train'][0], splits['target_train'][1])

    print("\n[Stage 1] Pretraining Source 6-layer CNN with early stopping...")
    source_model = CIFAR6CNN(num_classes=splits['num_source_classes']).to(device)
    optimizer = torch.optim.RMSprop(source_model.parameters(), lr=0.001, alpha=0.9)
    criterion = nn.CrossEntropyLoss()

    source_model, stage1_history, stage1_epochs = train_with_early_stopping(
        source_model,
        train_loader=loaders['source_train'],
        val_loader=loaders['source_test'],
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        max_epochs=pretrain_epochs,
        patience=early_stop_patience,
        target_val_loader=loaders['target_test'],
        target_dataset=target_dataset_full,
        sample_sizes=sample_sizes,
        num_replications=num_replications,
        prefix="Stage 1",
        verbose=True,
    )

    print(f"\n[Stage 1] Continuing for {post_early_stop_epochs} extra epochs after early stopping...")
    for ep in range(1, post_early_stop_epochs + 1):
        tr_loss = 0.0
        source_model.train()
        for x, y in loaders['source_train']:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = source_model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * x.size(0)
        avg_train_loss = tr_loss / len(loaders['source_train'].dataset)
        v_loss, v_acc = default_eval_loss(source_model, loaders['source_test'], criterion, device)

        target_val_loss_val = None
        pc_summary = ""
        if loaders['target_test'] is not None:
            target_val_loss_val, _ = default_eval_loss(
                source_model, loaders['target_test'], criterion, device
            )
            pc_results = _compute_pc_with_ci(
                source_model, target_dataset_full,
                sample_sizes=sample_sizes,
                num_replications=num_replications,
                device=device,
                base_seed=1000 + ep,
            )
            pc_summary = _summarize_pc(pc_results, CIFAR6CNN.LAYER_NAMES)

        log_parts = [
            f"Epoch {stage1_epochs + ep:02d}/{pretrain_epochs + post_early_stop_epochs:02d} (post-early-stop)",
            f"Train Loss: {avg_train_loss:.4f}",
            f"Source Test Acc: {v_acc * 100:.1f}%",
            f"Source Val Loss: {v_loss:.4f}",
        ]
        if target_val_loss_val is not None:
            log_parts.append(f"Target Val Loss: {target_val_loss_val:.4f}")
        if pc_summary:
            log_parts.append(f"PC: {pc_summary}")
        print("  " + " | ".join(log_parts))

        stage1_history.append({
            'epoch': stage1_epochs + ep,
            'train_loss': avg_train_loss,
            'val_loss': v_loss,
            'test_acc': v_acc * 100.0,
            'target_val_loss': target_val_loss_val,
            'pc_summary': pc_summary,
        })

    model_path = os.path.join(results_dir, f"pretrained_model_{ts}.pt")
    torch.save(source_model.state_dict(), model_path)
    print(f"Pretrained model saved to: {model_path}")

    stage1_path = os.path.join(results_dir, f"stage1_history_{ts}.json")
    _save_json(stage1_path, {'history': stage1_history})

    _plot_training_curves(
        stage1_history,
        os.path.join(results_dir, f"stage1_curves_{ts}.png"),
        title_suffix=" (Stage 1 - Source Pretraining)",
    )

    last_pc = stage1_history[-1].get('pc_summary', '')
    print(f"\n[Stage 1] Final PC profile: {last_pc}")

    print("\n[Stage 2] Fine-tuning on target dataset with early stopping...")
    target_model = create_transferred_model(
        pretrained_model=source_model,
        k=len(CIFAR6CNN.LAYER_NAMES),
    ).to(device)
    set_trainable_layers(target_model, trainable=True)

    target_optimizer = torch.optim.RMSprop(target_model.parameters(), lr=0.001, alpha=0.9)

    target_model, stage2_history, stage2_epochs = train_with_early_stopping(
        target_model,
        train_loader=loaders['target_train'],
        val_loader=loaders['target_test'],
        criterion=criterion,
        optimizer=target_optimizer,
        device=device,
        max_epochs=target_epochs,
        patience=early_stop_patience,
        target_val_loader=loaders['target_test'],
        target_dataset=target_dataset_full,
        sample_sizes=sample_sizes,
        num_replications=num_replications,
        prefix="Stage 2",
        verbose=True,
    )

    ft_model_path = os.path.join(results_dir, f"finetuned_model_{ts}.pt")
    torch.save(target_model.state_dict(), ft_model_path)
    print(f"Fine-tuned model saved to: {ft_model_path}")

    stage2_path = os.path.join(results_dir, f"stage2_history_{ts}.json")
    _save_json(stage2_path, {'history': stage2_history})

    _plot_training_curves(
        stage2_history,
        os.path.join(results_dir, f"stage2_curves_{ts}.png"),
        title_suffix=" (Stage 2 - Target Fine-tuning)",
    )

    print("\n" + "=" * 70)
    print("PRETRAINED EXPERIMENT SUMMARY")
    print("=" * 70)
    print(f"Case: {case.upper()}")
    print(f"Stage 1 epochs: {stage1_epochs} (+{post_early_stop_epochs} post-early-stop)")
    print(f"Stage 2 epochs: {stage2_epochs}")
    print(f"Results directory: {results_dir}")
    print(f"Timestamp: {ts}")
    print("=" * 70)

    return {
        'case': case,
        'sample_sizes': sample_sizes,
        'num_replications': num_replications,
        'stage1_epochs': stage1_epochs,
        'stage2_epochs': stage2_epochs,
        'post_early_stop_epochs': post_early_stop_epochs,
        'results_dir': results_dir,
        'timestamp': ts,
        'pretrained_model_path': model_path,
        'finetuned_model_path': ft_model_path,
        'stage1_history_path': stage1_path,
        'stage2_history_path': stage2_path,
    }



if __name__ == '__main__':


In [ ]:
parser = argparse.ArgumentParser(
        description="Pretrained transfer learning experiment (Di et al. 2025)."
    )
    parser.add_argument('--case', choices=['imbalance', 'balance'], default='imbalance')
    parser.add_argument('--sample_size', type=int, default=500,
                        help="Target sample size for PC profiling")
    parser.add_argument('--reps', type=int, default=2,
                        help="Number of PC replications for confidence intervals")
    parser.add_argument('--pretrain_epochs', type=int, default=150,
                        help="Max pretraining epochs (source)")
    parser.add_argument('--target_epochs', type=int, default=100,
                        help="Max fine-tuning epochs (target)")
    parser.add_argument('--patience', type=int, default=EARLY_STOP_PATIENCE,
                        help="Early stopping patience")
    parser.add_argument('--post_epochs', type=int, default=POST_EARLY_STOP_EPOCHS,
                        help="Extra epochs after early stopping in Stage 1")
    parser.add_argument('--real_cifar', action='store_true',
                        help="Use real CIFAR-10 if available")
    parser.add_argument('--max_images', type=int, default=None,
                        help="Limit number of CIFAR-10 images (debug)")
    parser.add_argument('--data_dir', type=str, default='./data',
                        help="Folder with CIFAR-10 data")
    parser.add_argument('--no_hf', action='store_true',
                        help="Do not use HuggingFace datasets")
    args = parser.parse_args()

    run_pretrained_experiment(
        case=args.case,
        sample_sizes=[args.sample_size],
        num_replications=args.reps,
        pretrain_epochs=args.pretrain_epochs,
        target_epochs=args.target_epochs,
        use_synthetic=not args.real_cifar,
        max_images=args.max_images,
        data_dir=args.data_dir,
        use_hf=not args.no_hf,
        early_stop_patience=args.patience,
        post_early_stop_epochs=args.post_epochs,
    )
